# GND Data Processing and Matching Tool
This notebook contains a workflow to parse MARCXML records from the German National Library (GND), clean entity name variants, and perform matching against local datasets.

In [ ]:
# Section 1: Read and Inspect MARCXML
from pymarc import parse_xml_to_array

# Set your local path to the GND MARCXML export
file_path = "data/gnd_authority_records.xml"

with open(file_path, "rb") as fh:
    records = parse_xml_to_array(fh)

# Inspect the first 100 records
for i, record in enumerate(records[:100]):
    print(f"\n--- Record {i + 1} ---")
    try:
        title = record.title or "(No title)"
        print(f"Title: {title}")
    except Exception as e:
        print(f"Title: [Error reading title] {e}")
    
    print(f"Leader: {record.leader}")
    try:
        for field in record.get_fields():
            print(field)
    except Exception as e:
        print(f"[Error printing fields] {e}")

In [ ]:
# Section 2: Filter GND List (e.g., vif, level 1)
import csv
from pymarc import parse_xml_to_array

input_file = "data/gnd_authority_records.xml"
output_csv = "data/gnd_filtered_records.csv"

with open(input_file, "rb") as fh:
    records = parse_xml_to_array(fh)

filtered_rows = []

for record in records:
    # Check for field 075 subfield $b == 'vif' (Entity type)
    is_vif = any(
        field.tag == "075" and 'vif' in field.get_subfields('b')
        for field in record.get_fields("075")
    )

    # Check for 042 $a (Quality level/Code)
    gnd_qualifiers = {"gnd1", "gnd2", "gnd3"}
    has_gnd_code = any(
        field.tag == "042" and
        any(code in gnd_qualifiers for code in field.get_subfields('a'))
        for field in record.get_fields("042")
    )

    if is_vif and has_gnd_code:
        gnd_id = record['001'].value() if record['001'] else ''

        # Extract Field 111 (Preferred Name of Conference/Event)
        field_111 = ""
        for field in record.get_fields("111"):
            subfields = field.get_subfields('a', 'b', 'c', 'd', 'e', 'n', 'q')
            if subfields:
                field_111 = " ".join(subfields)
                break

        # Extract Field 411 (Variant Names)
        field_411_list = []
        for field in record.get_fields("411"):
            subfields = field.get_subfields('a', 'b', 'c', 'd', 'e', 'n', 'q')
            if subfields:
                name_variant = " ".join(subfields)
                field_411_list.append(name_variant.strip())

        filtered_rows.append({
            'GND-ID': gnd_id,
            'Field 111': field_111,
            'Field 411': " | ".join(field_411_list),
        })

with open(output_csv, mode='w', encoding='utf-8', newline='') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=['GND-ID', 'Field 111', 'Field 411'])
    writer.writeheader()
    writer.writerows(filtered_rows)

print(f"✅ Exported {len(filtered_rows)} records to: {output_csv}")

In [ ]:
# Section 3: Clean Name Variants in Local Dataset
import pandas as pd

input_excel = "data/local_entities.xlsx"
df = pd.read_excel(input_excel)

def split_name_variants(cell):
    if pd.isna(cell):
        return []
    return [name.strip() for name in str(cell).split(",") if name.strip()]

column_name = "Abweichende Benennungen"
df["Name Variants List"] = df[column_name].apply(split_name_variants)

output_path = "data/cleaned_local_variants.xlsx"
df.to_excel(output_path, index=False)
print("✅ Local variants cleaned.")

In [ ]:
# Section 4: Clean Name Variants in GND Export
import pandas as pd
import re

csv_path = "data/gnd_filtered_records.csv"
df = pd.read_csv(csv_path, dtype=str)

def split_variants(text):
    if pd.isna(text):
        return []
    # Split on pipes or commas
    parts = re.split(r'[|,]', text)
    cleaned = {part.strip() for part in parts if part.strip()}
    return list(cleaned)

df["Field 411 Variants"] = df["Field 411"].apply(split_variants)

output_path = "data/gnd_filtered_records_cleaned.csv"
df.to_csv(output_path, index=False)
print("✅ GND variants cleaned.")

In [ ]:
# Section 5: Run Full Matching (All Matches)
import pandas as pd
from collections import defaultdict
from tqdm import tqdm
import ast

local_path = "data/cleaned_local_variants.xlsx"
gnd_path = "data/gnd_filtered_records_cleaned.csv"
output_path = "data/all_matches_results.xlsx"

df_excel = pd.read_excel(local_path)
df_csv = pd.read_csv(gnd_path, dtype=str)

# Convert string representations of lists back to list objects
df_excel["Name Variants List"] = df_excel["Name Variants List"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else []
)

# Pre-processing for faster lookup
field_111_lookup = defaultdict(list)
variant_to_row_411 = defaultdict(list)

df_csv["Field 111_norm"] = df_csv["Field 111"].fillna("").str.strip().str.lower()
for _, row in df_csv.iterrows():
    field_111_lookup[row["Field 111_norm"]].append(row)
    # Process variant sub-list
    vars_list = ast.literal_eval(row["Field 411 Variants"]) if isinstance(row["Field 411 Variants"], str) else []
    for v in vars_list:
        v_norm = v.strip().lower()
        if len(v_norm) >= 3:
            variant_to_row_411[v_norm].append(row)

results = []
# ... (Matching logic continues using normalized lookups)
print("✅ Matching logic prepared for execution.")